# Lab 1: breast cancer diagnosis end to end

Day 1 practice, 2 hours.

Goal: build a leakage-free ML pipeline, compare models, and choose an operating point for
high sensitivity. Section 0 is a 15 minute warm-up on the other half of supervised learning,
regression, so that both task types are in your hands before the project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold,
                                     cross_val_score, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (confusion_matrix, accuracy_score, roc_auc_score,
                             mean_absolute_error, ConfusionMatrixDisplay, RocCurveDisplay)

RANDOM_STATE = 42

## 0. Warm-up: regression vs classification

Same recipe, different target. Here the target is a continuous measure of diabetes
progression one year after baseline, so the error is measured in the units of the outcome
rather than as a count of mistakes.

Note the split of duties: `KFold` for regression, `StratifiedKFold` for classification.

In [ ]:
Xr, yr = load_diabetes(return_X_y=True, as_frame=True, scaled=False)
print(Xr.shape, 'target range', round(yr.min(), 1), 'to', round(yr.max(), 1))

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# TODO: compare DummyRegressor(strategy='mean') with LinearRegression using
#       cross_val_score(..., cv=kf, scoring='neg_mean_absolute_error')
# TODO: how many progression units is the linear model better than predicting the mean?
# TODO: would you deploy a model with that MAE? What would a clinician need to know first?

## 1. Load data

Important: in scikit-learn, `0 = malignant` and `1 = benign`. For medical evaluation we make
malignant the positive class, because the positive class is the one we are trying not to miss.

In [ ]:
X, y_raw = load_breast_cancer(return_X_y=True, as_frame=True)
y = 1 - y_raw  # TODO: explain why we do this
print(X.shape)
print(pd.Series(y).value_counts(normalize=True).rename({0: 'benign', 1: 'malignant'}))

## 2. EDA

In [ ]:
# TODO: show describe()
# TODO: plot histograms for three features, separated by class
# TODO: inspect correlation among radius, perimeter, area

## 3. Split and baseline

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

baseline = DummyClassifier(strategy='most_frequent')
# TODO: fit baseline, print accuracy and confusion matrix
# TODO: the baseline accuracy is already high. Why is that not reassuring?

## 4. Compare real models with stratified CV

Report mean plus or minus standard deviation. A gap of 0.01 between two models whose
standard deviation is 0.03 is not a result.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'logreg': Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(max_iter=2000))]),
    'knn_5': Pipeline([('scale', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=5))]),
    'tree_depth_3': DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
}

# TODO: compute ROC-AUC mean and std for each model
# TODO: try k in {1,3,5,11,21} and tree depth 1..10, plot train vs CV score against depth
# TODO: at which depth does the tree start memorising rather than learning?

## 5. Choose a threshold without test leakage

Choose the threshold on the training set using cross-validated probabilities. Then apply it
once to the test set. Choosing it on the test set is the most common way to report a model
that will not reproduce in a hospital.

In [ ]:
best_model = models['logreg']
# TODO: use cross_val_predict(..., method='predict_proba') on X_train only
# TODO: choose the threshold that maximises specificity subject to sensitivity >= 0.95
# TODO: fit best_model on all train data, apply that threshold to test probabilities
# TODO: report sensitivity, specificity, ROC-AUC, confusion matrix

## 6. Short written answers

1. Why is accuracy dangerous here?
2. What clinical decision could this model support, and who is the user?
3. Which error is worse here, a false negative or a false positive? Why?
4. Your threshold gives sensitivity 0.95 on this dataset. What happens to PPV if the same
   model is used for population screening where prevalence is 1 percent?